In [ ]:
import os 

from dotenv import load_dotenv
load_dotenv()



KAGGLE_USERNAME = os.getenv('KAGGLE_USERNAME')
KAGGLE_KEY = os.getenv('KAGGLE_KEY')




In [ ]:
!kaggle datasets download -d ankitbansal06/retail-orders

In [ ]:
#!unzip retail-orders.zip -d orders

In [ ]:
import pandas as pd

df = pd.read_csv('orders/orders.csv')

In [ ]:
df.head(30)

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
for col in df.select_dtypes(include = object):
    if col != 'Order Date':
        print(f'Unique values in {col}')
        print(df[col].unique())


In [ ]:
# Replace Invalids: In pandas, replace the entries 'Not Available' and 'unknown' in the Ship Mode column with NaN.
import numpy as np
df['Ship Mode']= df['Ship Mode'].replace(['Not Available','unknown'], np.nan)


print(df['Ship Mode'].unique())

In [ ]:
# 4.3. Rename Columns: Show how to rename all DataFrame columns by lowercasing and replacing spaces with underscores.

df.columns = df.columns.str.lower().str.replace(' ','_')

print(df.columns)

In [ ]:
df['discout'] = df['list_price'] * df['discount_percent']/100

df[['list_price','discount_percent','discout']].head()

In [ ]:
df['selling_price'] = df['list_price'] - df['discout']
df[['list_price','discout','selling_price']].head()

In [ ]:
df['profit']=df['selling_price']-df['cost_price']
df[['selling_price','cost_price','profit']].head()

In [ ]:

df.drop(['cost_price','list_price','discount_percent'],axis = 1,inplace=True)


In [ ]:
df.head()

In [ ]:


df['order_date'] = pd.to_datetime(df['order_date'],format = 'ISO8601')
df

### Database Operations

In [ ]:
# Connection String: Using environment variables HOST, DATABASE, USER, PASSWORD,and PORT, construct the SQLAlchemy connection string for a MySQL database

!pip install sqlalchemy

In [ ]:
!pip install pymysql

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()

DATABASE = os.getenv('DATABASE')
USER = os.getenv('USER')
PASSWORD = os.getenv('PASSWORD')
HOST = os.getenv('HOST')
PORT = os.getenv('PORT')


In [ ]:
# connection string
from sqlalchemy import create_engine

engine = create_engine(f'mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}',echo = True)

In [ ]:
!pip install cryptography

In [ ]:
try:
    with engine.connect() as connection:
        print('Connection is sucessful!')
except Exception as e:
    print('Connection failed',e)

In [ ]:
# Write to SQL: Write the DataFrame into a table named orders in the connected database, appending if the table exists and omitting the index.
df.to_sql('orders',con=engine,if_exists='replace',index=False)

In [ ]:
# Read All Rows: Write a SQLAlchemy query to select all rows from orders, execute it,and fetch all results.
from sqlalchemy import text

o = text('SELECT * FROM orders')

with engine.connect() as conn:
    result = conn.execute(o)
    all_rows = result.fetchall()
    for row in all_rows[:5]:
        print(row)

### SQL Analysis Queries

In [ ]:
# Total Revenue per Category: Sum selling_price grouped by category, ordered descending

# df.columns
Total_Rev = df.groupby('category')['selling_price'].sum().sort_values(ascending=False)

print(Total_Rev)

In [ ]:
# Top 3 Profitable Cities: Identify the three cities with the highest total profit
#df.columns
highest_total_profit_cities = df.groupby('city')['profit'].sum().sort_values(ascending=False).head(3)

print(highest_total_profit_cities)

In [ ]:
#Profit Margin per Product: Compute (SUM(profit)/SUM(selling_price))*100 per product_id, list the top 25.
# df.columns


df_margin = df.groupby('product_id').agg({'profit':'sum','selling_price':'sum'})


df_margin['margin_percent']=df_margin['profit']/df_margin['selling_price'] * 100
# print(df_margin)

df_margin['margin_percent'].sort_values(ascending=False).head(25)


In [ ]:
# Order Counts by Shipping Mode: Count order_id per ship_mode, ordered descending.
#df.head()


order_counts = df.groupby('ship_mode')['order_id'].count().sort_values(ascending=False)

print(order_counts)

In [ ]:
# Month with Highest Orders: Find the month number (MONTH(order_date)) with the greatest count of order_id.


month_with_highest_orders = df.groupby((df['order_date']).dt.month)['order_id'].count().sort_values(ascending=False)
print(month_with_highest_orders)


In [ ]:
# Top 5 Most Discounted Products: Select product_id and discount, ordered by discount descending.

top_5_discounted_products = df[['product_id','discout']].sort_values('discout',ascending = False).head(5)

print(top_5_discounted_products)

In [ ]:
# Top 3 Cities by Sales: Sum selling_price per city, list the top three
top_3_cities = df.groupby('city')['selling_price'].sum().sort_values(ascending=False).head(3)

print(top_3_cities)

In [ ]:
# Average Profit for 2023: Calculate the average profit for orders where YEAR(order_date)=2023

avg_profit_2023 = df[df['order_date'].dt.year == 2023]['profit'].mean()

print(avg_profit_2023)

In [ ]:
# Sub-Category Sales Ranking: Sum selling_price by sub_category, show the top five.

sub_category_ranking = df.groupby('sub_category')['selling_price'].sum().sort_values(ascending=False).head(5)

print(sub_category_ranking)

In [ ]:
# Average Selling Price per Category: Compute average selling_price for each category, ordered descending

avg_category_ranking = df.groupby('category')['selling_price'].sum().sort_values(ascending=False)

print(avg_category_ranking)

In [ ]:
#Top 10 Products by Order Count: Count order_id per product_id, list the top ten.

top_10_products = df.groupby('product_id')['order_id'].count().sort_values(ascending=False).head(10)

print(top_10_products)

In [ ]:
# Top 5 Profitable Products: Sum profit per product_id, list the top five.
top_5_profitable_products = df.groupby('product_id')['profit'].sum().sort_values(ascending=False).head(5)

print(top_5_profitable_products)

In [ ]:
# Year with Highest Sales: Find YEAR(order_date) with maximum sum of selling_price.

highest_sale_year = df.groupby(df['order_date'].dt.year)['selling_price'].sum().sort_values(ascending=False).head(1)

print(highest_sale_year)

In [ ]:
# Region with Lowest Discount: Identify region with the lowest average discount.

lowest_discount_region = df.groupby('region')['discout'].mean().sort_values(ascending=True).head(1)

print(lowest_discount_region)

In [ ]:
#  Yearly Sales Growth: Show total sales (SUM(selling_price)) per year, ordered by year.

yearly_sale_growth = df.groupby(df['order_date'].dt.year)['selling_price'].sum().sort_values(ascending=False)

print(yearly_sale_growth)

In [ ]:
# Profit Contribution by Category: For each category, compute SUM(profit)*100 / (SELECT SUM(profit) FROM orders).

total_profit = df['profit'].sum()

profit_contribution_by_category = df.groupby('category')['profit'].sum() * 100 / total_profit

print(profit_contribution_by_category)

In [ ]:
# Most Common Shipping Mode: Find the single ship_mode used by the most orders.

most_common_ship_mode = df.groupby('ship_mode')['order_id'].count().sort_values(ascending=False).head(1)

print(most_common_ship_mode)

In [ ]:
# Highest Average Order Value by Region: Determine which region has the highest average selling_price

highest_avg_order_value = df.groupby('region')['selling_price'].mean().sort_values(ascending=False).head(1)

print(highest_avg_order_value)

In [ ]:
# Category & Sub-Category Sales: Sum selling_price for each (category,sub_category) pair.

cat_sub_cat_sales = df.groupby(['category','sub_category'])['selling_price'].sum()

print(cat_sub_cat_sales )

In [ ]:
#  Top 3 Profitable Products per Sub-Category: For each sub_category, list the top three product_id by total profit.

top_3_profitable_products = df.groupby(['sub_category','product_id'])['profit'].sum()

print(top_3_profitable_products)

In [ ]:
# Monthly Order Counts for 2023: Count order_id by MONTH(order_date) for YEAR(order_date)=2023, list in descending order.

monthly_orders_counts_2023 = (df[df["order_date"].dt.year == 2023].groupby(df["order_date"].dt.month)["order_id"].count().sort_values(ascending=False))


print(month_with_highest_orders)

In [ ]:
# States with Lowest Sales: Identify the ten state values with the smallest total selling_price.

lowest_sales_states = df.groupby('state')['selling_price'].sum().sort_values(ascending=True).head(10)

print(lowest_sales_states)

In [ ]:
#  Top 5 States by Revenue: Find the five state values contributing most to total revenue.
highest_5_sales_states = df.groupby('state')['selling_price'].sum().sort_values(ascending=False).head(5)

print(highest_5_sales_states)

In [ ]:
# Average Discount by Sub-Category: Compute average discount per sub_category, ordered descending.
avg_disc_sub_category= df.groupby('sub_category')['discout'].mean().sort_values(ascending=False)

print(avg_disc_sub_category)

In [ ]:
# Top 10 Products by Revenue: Sum selling_price per product_id, list the top ten by total revenue.
top_10_products_by_revenue = df.groupby('product_id')['selling_price'].sum().sort_values(ascending=False).head(10)

print(top_10_products_by_revenue)